In [9]:
import pandas as pd
import numpy as np
import os
import sys
import re

#해당 디렉토리 내부에 있는 모든 디렉토리의 이름을 가져온다
def get_dir_list(path):
    dir_list = []
    for root, dirs, files in os.walk(path):
        for dir in dirs:
            dir_list.append(dir)
    return dir_list

def get_file_list(dir_path):
    file_list = []
    for root, dirs, files in os.walk(dir_path):
        for file in files:
            file_list.append(file)
    return file_list

def parse_performance(log_str):
    results = {}
    sections = ['Validation', 'Test']

    for section in sections:
        pattern = rf"{section} - Time: ([\d.]+).*?Recall@20: ([\d.]+), NDCG@20: ([\d.]+);.*?Recall@50: ([\d.]+), NDCG@50: ([\d.]+);"
        match = re.search(pattern, log_str, re.DOTALL)
        if match:
            results[section.lower()] = {
                'recall@20': float(match.group(2)),
                'ndcg@20': float(match.group(3)),
                'recall@50': float(match.group(4)),
                'ndcg@50': float(match.group(5))
            }
    return results

def analyze_results(dir_path):
    file_list = get_file_list(dir_path)
    log_list = []
    for file in file_list:
        file_path = dir_path + '/' + file
        with open(file_path) as f:
            logs = "".join(f.readlines()[-8:])
        parsed = parse_performance(logs)
        log_list.append(parsed)
    return log_list
        

In [10]:
target_dataset = "Yelp"
target_model = "LightGCN"
target_method = "VQ"
target_dir = f"../log/{target_dataset}/{target_model}/{target_method}/"
dir_list = get_dir_list(target_dir)

import csv

with open(f"./csv/summary_{target_dataset}_{target_model}_{target_method}.tsv", mode='w', newline='') as file:
    cw = csv.writer(file, delimiter='\t')
    cw.writerow(['dir_name', 'mean_valid_recall@20', 'mean_valid_recall@50', 'mean_valid_ndcg@20', 'mean_valid_ndcg@50', 'mean_test_recall@20', 'mean_test_recall@50', 'mean_test_ndcg@20', 'mean_test_ndcg@50'])
    for dir_name in dir_list:
        dir_path = os.path.join(target_dir, dir_name)
        log_list = analyze_results(dir_path)
        mean_valid_recall_20 = np.mean([log['validation']['recall@20'] for log in log_list]).round(4)
        mean_valid_recall_50 = np.mean([log['validation']['recall@50'] for log in log_list]).round(4)
        mean_valid_ndcg_20 = np.mean([log['validation']['ndcg@20'] for log in log_list]).round(4)
        mean_valid_ndcg_50 = np.mean([log['validation']['ndcg@50'] for log in log_list]).round(4)
        mean_test_recall_20 = np.mean([log['test']['recall@20'] for log in log_list]).round(4)
        mean_test_recall_50 = np.mean([log['test']['recall@50'] for log in log_list]).round(4)
        mean_test_ndcg_20 = np.mean([log['test']['ndcg@20'] for log in log_list]).round(4)
        mean_test_ndcg_50 = np.mean([log['test']['ndcg@50'] for log in log_list]).round(4)
        cw.writerow([dir_name, mean_valid_recall_20, mean_valid_recall_50, mean_valid_ndcg_20, mean_valid_ndcg_50, mean_test_recall_20, mean_test_recall_50, mean_test_ndcg_20, mean_test_ndcg_50])  






